In [ ]:
!pip install tensorflow

In [ ]:
# import pertinent libraries
import os
import sys
import datetime
import glob as glob
import numpy as np
import cv2
# [Keras Models]
# import the Keras implementations of VGG16, VGG19, InceptionV3 and Xception models
# the model used here is VGG16
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D
from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from keras.optimizers import SGD
import tensorflow
from scipy.interpolate import interp1d
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from keras.applications.vgg19 import VGG19

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/classify/'

In [ ]:
train_dir = '/content/drive/MyDrive/classify/'
validate_dir = '/content/drive/MyDrive/test'

In [ ]:

# กำหนดค่า
img_width, img_height = 180, 180
nb_epochs = 10
batch_size = 16
nb_classes = 20


# นับจำนวนภาพในชุดข้อมูลการฝึก
nb_train_samples = sum([len(files) for r, d, files in os.walk(train_dir)])
# นับจำนวนภาพในชุดข้อมูลการตรวจสอบ
nb_validate_samples = sum([len(files) for r, d, files in os.walk(validate_dir)])

# คำนวณจำนวนขั้นตอนต่อ epoch
steps_per_epoch = nb_train_samples // batch_size
validation_steps = nb_validate_samples // batch_size

# data pre-processing for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    fill_mode='nearest',
    horizontal_flip=True)

# data pre-processing for validation
validate_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    fill_mode='nearest',
    horizontal_flip=True)

# generate and store training data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical')  # Ensure that class_mode is set

# generate and store validation data
validate_generator = validate_datagen.flow_from_directory(
    validate_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical')  # Ensure that class_mode is set

# set up transfer learning on pre-trained ImageNet VGG19 model
vgg19_model = VGG19(weights='imagenet', include_top=False)
x = vgg19_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(nb_classes, activation='softmax')(x)
model = Model(inputs=vgg19_model.input, outputs=predictions)
model.summary()

# freeze all layers of the pre-trained VGG19 model
for layer in vgg19_model.layers:
    layer.trainable = False

# compile the new model using a RMSProp optimizer
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print('Number of training samples:', nb_train_samples)
print('Number of validation samples:', nb_validate_samples)

# fit the model, log the results and the training time
now = datetime.datetime.now
t = now()
transfer_learning_history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=nb_epochs,
    validation_data=validate_generator,
    validation_steps=validation_steps)
print('Training time: %s' % (now() - t))


Found 2477 images belonging to 20 classes.
Found 2105 images belonging to 20 classes.
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, None, None, 3)]   0         
                                                                 
 block1_conv1 (Conv2D)       (None, None, None, 64)    1792      
                                                                 
 block1_conv2 (Conv2D)       (None, None, None, 64)    36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, None, None, 64)    0         
                                                                 
 block2_conv1 (Conv2D)       (None, None, None, 128)   73856     
                                                                 
 block2_conv2 (Conv2D)       (None, None, None, 128)   147584    
                                       

156/156 [==============================] - 1501s 10s/step - loss: 1.8624 - accuracy: 0.4562 - val_loss: 2.2326 - val_accuracy: 0.3044
Training time: 0:25:02.107722


In [ ]:
# ประมวลผลการทำนายบนชุดข้อมูลตรวจสอบ
validate_generator.reset()
predictions = model.predict(validate_generator, steps=validation_steps, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

# รับค่าจริงจากชุดข้อมูลตรวจสอบ
true_classes = validate_generator.classes
class_labels = list(validate_generator.class_indices.keys())



131/131 [==============================] - 665s 5s/step


NameError: name 'confusion_matrix' is not defined

In [ ]:
print("True classes shape:", true_classes.shape)
print("Predicted classes shape:", predicted_classes.shape)

# ตรวจสอบข้อมูลว่าตรงกันหรือไม่
if len(true_classes) != len(predicted_classes):
    raise ValueError("The length of true classes and predicted classes does not match.")


True classes shape: (2105,)
Predicted classes shape: (2096,)


ValueError: The length of true classes and predicted classes does not match.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# สร้าง Confusion Matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)

# แสดงผล Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d',
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# แสดงผล classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print(report)

ValueError: Found input variables with inconsistent numbers of samples: [2105, 2096]

In [ ]:
# [Dataset]
# image dimensions for VGG16, VGG19 are 224, 224
# image dimensions for InceptionV3 and Xception are 299, 299
img_width, img_height = 180, 180

# train_dir = train_dir.repeat()
# validate_dir = validate_dir.repeat()
nb_epochs = 20
batch_size = 32
nb_classes = 20
# nb_classes = len(glob.glob(train_dir + '/*'))

# get number of images in training directory
nb_train_samples = 0
for r, dirs, files in os.walk(train_dir):
    for dr in dirs:
        nb_train_samples += len(glob.glob(os.path.join(r, dr + "/*")))
# get number of images in validation directory
nb_validate_samples = 0
for r, dirs, files in os.walk(validate_dir):
    for dr in dirs:
        nb_validate_samples += len(glob.glob(os.path.join(r, dr + "/*")))

In [ ]:
# data pre-processing for training
train_datagen =  ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    fill_mode = 'nearest',
    horizontal_flip = True)

# data pre-processing for validation
validate_datagen =  ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    fill_mode = 'nearest',
    horizontal_flip = True)

In [ ]:
# generate and store training data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (img_width, img_height),
    batch_size = batch_size)

# generate and store validation data
validate_generator = validate_datagen.flow_from_directory(
    validate_dir,
    target_size = (img_width, img_height),
    batch_size = batch_size)

Found 2477 images belonging to 20 classes.
Found 2105 images belonging to 20 classes.


In [ ]:
# set up transfer learning on pre-trained ImageNet VGG19 model - remove fully connected layer and replace
# with softmax for classifying the number of classes in the dataset
vgg19_model = VGG19(weights = 'imagenet', include_top = False)
x = vgg19_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(nb_classes, activation = 'softmax')(x)
model = Model(inputs = vgg19_model.input, outputs = predictions)

80134624/80134624 [==============================] - 1s 0us/step


In [ ]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, None, None, 3)]   0         
                                                                 
 block1_conv1 (Conv2D)       (None, None, None, 64)    1792      
                                                                 
 block1_conv2 (Conv2D)       (None, None, None, 64)    36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, None, None, 64)    0         
                                                                 
 block2_conv1 (Conv2D)       (None, None, None, 128)   73856     
                                                                 
 block2_conv2 (Conv2D)       (None, None, None, 128)   147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, None, None, 128)   0     

In [ ]:
# freeze all layers of the pre-trained InceptionV3 model
for layer in vgg19_model.layers:
    layer.trainable = False

In [ ]:
# compile the new model using a RMSProp optimizer
model.compile(optimizer = 'rmsprop',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy'])

In [ ]:
nb_train_samples = sum([len(files) for r, d, files in os.walk(train_dir)])
nb_validate_samples = sum([len(files) for r, d, files in os.walk(validate_dir)])

print('Number of training samples:', nb_train_samples)
print('Number of validation samples:', nb_validate_samples)

Number of training samples: 2505
Number of validation samples: 2107


In [ ]:
steps_per_epoch = nb_train_samples // batch_size

validation_steps = nb_validate_samples // batch_size

In [ ]:
print(batch_size, steps_per_epoch, validation_steps)

32 78 65


In [ ]:
# fit the model, log the results and the training time
now = datetime.datetime.now
t = now()
transfer_learning_history = model.fit(
    train_generator,
    epochs=15 ,
    validation_data = validate_generator)
print('Training time: %s' % (now() - t))

NameError: name 'steps_per_epoch' is not defined